In [1]:
import os

import pandas as pd
import numpy as np
import json

In [2]:
os.getcwd()

'/Users/yilingyun/Documents/WorkInProgress/Charade/latent-force-model-project/human/behavioralExpDataAndAnalysis/exp2'

# Helper functions

In [3]:
def find_exptVer(row, studyVer):
    if studyVer == "offdiagonal":
        if row.subjNum < 400:
            EXPT_N = 65 
            exptVer = row.subjNum % EXPT_N
            if (exptVer == 0):
                exptVer = EXPT_N
        elif row.subjNum <= 554:
            EXPT_START = 66
            EXPT_END = 211
            EXPT_N = EXPT_END - EXPT_START + 1  # 146
            exptVer = (row.subjNum % EXPT_N) + EXPT_START 
        else:
            EXP_VERSIONS = [71, 72, 73, 78, 82,
                            85, 86, 93, 96, 119,
                            126, 128, 129, 149, 159,
                            164, 173, 174, 184, 193,
                            199, 204, 206, 211]
            N_TEST = len(EXP_VERSIONS)
            idx = (row.subjNum - 1) % N_TEST
            exptVer = EXP_VERSIONS[idx]
    elif studyVer == "stimset1" or studyVer == "stimset2":
        EXPT_N = 65
        exptVer = row.subjNum % EXPT_N

        if (exptVer == 0):
            exptVer = EXPT_N
    return exptVer


In [4]:
def find_exptOptions(row, studyVer):
    exptVer = row.exptVer
    if studyVer == "offdiagonal":
        jsonName = './../../behavioralExpCode/exp2/offdiagonal/input/expt' + str(exptVer) + '.json'
    elif studyVer == "stimset1":
        jsonName = './../../behavioralExpCode/exp2/stimSet1/input/expt' + str(exptVer) + '.json'
    elif studyVer == "stimset2":
        jsonName = './../../behavioralExpCode/exp2/stimSet2/input/expt' + str(exptVer) + '.json'

    with open(jsonName, 'r') as f:
        data = json.load(f)
    return data[str(row.exptId)]

In [5]:
def find_choice(row):
    if row.choicePos == "left":
        return row.options[0]
    if row.choicePos == "middle":
        return row.options[1]
    if row.choicePos == "right":
        return row.options[2]

# StimSet1

## Subj file

In [6]:
subj_stimset1 = pd.read_csv("./raw/subj_HSvideo_stimset1.txt", delimiter="\t")
subj_stimset1.head()

,num,date,startTime,id,userAgent,endTime,duration,quizAttemptN,instrReadingTimes,quickReadingPageN,...,hiddenDurations,comments,serious,maximized,problems,gender,age,inView,viewportW,viewportH
0,1,2024-10-08,23:16:07,67846,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,23:35:53,19.752700,1,"{""0"":3.921,""1"":6.676,""2"":10.601,""3"":15.68,""4"":...",0,...,NaN,NaN,1,1,N/A. I just had difficulty finding the odd one...,M,20,True,1536,864
1,3,2024-10-08,23:19:39,67851,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,23:59:15,39.596517,1,"{""0"":0.956,""1"":1.671,""2"":2.621,""3"":7.022,""4"":1...",0,...,NaN,NaN,1,1,None of it was unclear but at times it was har...,M,18,True,1440,900
2,4,2024-10-08,23:25:15,67998,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,00:08:53,43.629583,1,"{""0"":1.138,""1"":3.412,""2"":5.454,""3"":8.593,""4"":1...",0,...,2.545,NaN,1,1,NaN,F,18,True,1440,900
3,6,2024-10-08,23:57:45,63805,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,00:24:21,26.614033,1,"{""0"":4.676,""1"":7.235,""2"":71.03,""3"":79.243,""4"":...",0,...,NaN,NaN,1,1,NaN,F,20,True,1512,945
4,7,2024-10-09,00:08:22,67795,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,00:34:07,25.745483,1,"{""0"":32.602,""1"":36.445,""2"":38.315,""3"":43.502,""...",0,...,"0.699,0.019,13.738",NaN,1,1,No,F,21,True,1470,919


In [7]:
subj_stimset1.shape

(73, 21)

In [8]:
len(subj_stimset1.loc[subj_stimset1.serious == 0])

7

In [9]:
len(subj_stimset1.loc[subj_stimset1.maximized == 0]) 

2

In [10]:
excludedSubjNum_stimset1 = pd.concat([subj_stimset1.loc[subj_stimset1.serious == 0, "num"], subj_stimset1.loc[subj_stimset1.maximized == 0, "num"]])
len(np.unique(excludedSubjNum_stimset1))

8

## Trial file

In [11]:
trial_stimset1 = pd.read_csv("./raw/trial_HSvideo_stimset1.txt", delimiter=";")
trial_stimset1.head()

,subjNum,subjStartDate,subjStartTime,trialIndex,exptId,choicePos,vidPlayCounts,rt
0,1,2024-10-08,23:16:07,0,29,right,"{""vid1"":1,""vid2"":1,""vid3"":1}",21.897
1,1,2024-10-08,23:16:07,1,34,right,"{""vid1"":2,""vid2"":2,""vid3"":2}",39.827
2,2,2024-10-08,23:17:20,0,7,middle,"{""vid1"":1,""vid2"":2,""vid3"":1}",13.386
3,1,2024-10-08,23:16:07,2,19,middle,"{""vid1"":3,""vid2"":3,""vid3"":3}",58.960
4,1,2024-10-08,23:16:07,3,39,middle,"{""vid1"":4,""vid2"":4,""vid3"":4}",82.882


In [12]:
trial_stimset1.shape

(3369, 8)

In [13]:
incompleteSubjNum_stimset1 = trial_stimset1.subjNum.value_counts().loc[lambda x : x < 45].index

In [14]:
completeSubj_stimset1 = subj_stimset1[~subj_stimset1.num.isin(incompleteSubjNum_stimset1)]
cleanedSubj_stimset1 = completeSubj_stimset1[~completeSubj_stimset1.num.isin(excludedSubjNum_stimset1)]

In [15]:
completeTrial_stimset1 = trial_stimset1[~trial_stimset1.subjNum.isin(incompleteSubjNum_stimset1)]
completeTrial_stimset1.head()

,subjNum,subjStartDate,subjStartTime,trialIndex,exptId,choicePos,vidPlayCounts,rt
0,1,2024-10-08,23:16:07,0,29,right,"{""vid1"":1,""vid2"":1,""vid3"":1}",21.897
1,1,2024-10-08,23:16:07,1,34,right,"{""vid1"":2,""vid2"":2,""vid3"":2}",39.827
3,1,2024-10-08,23:16:07,2,19,middle,"{""vid1"":3,""vid2"":3,""vid3"":3}",58.960
4,1,2024-10-08,23:16:07,3,39,middle,"{""vid1"":4,""vid2"":4,""vid3"":4}",82.882
6,1,2024-10-08,23:16:07,4,31,right,"{""vid1"":5,""vid2"":5,""vid3"":5}",106.792


In [16]:
#drop not serious or not maximized
excSubjTrial_stimset1 = completeTrial_stimset1[~completeTrial_stimset1.subjNum.isin(excludedSubjNum_stimset1)]
len(np.unique(excSubjTrial_stimset1.subjNum)) #N

65

In [17]:
excSubjTrial_stimset1 = excSubjTrial_stimset1.copy()
excSubjTrial_stimset1["exptVer"] = excSubjTrial_stimset1.apply(lambda row: find_exptVer(row, studyVer="stimset1"), axis = 1)
excSubjTrial_stimset1["options"] = excSubjTrial_stimset1.apply(lambda row: find_exptOptions(row, studyVer="stimset1"), axis = 1) 
excSubjTrial_stimset1["choice"] = excSubjTrial_stimset1.apply(lambda row: find_choice(row), axis = 1)
cleanedTrial_stimset1 = excSubjTrial_stimset1.reset_index(drop = True)
cleanedTrial_stimset1

,subjNum,subjStartDate,subjStartTime,trialIndex,exptId,choicePos,vidPlayCounts,rt,exptVer,options,choice
0,1,2024-10-08,23:16:07,0,29,right,"{""vid1"":1,""vid2"":1,""vid3"":1}",21.897,1,"[5570_poke, 4806_flirt with, 5901_huddle with]",5901_huddle with
1,1,2024-10-08,23:16:07,1,34,right,"{""vid1"":2,""vid2"":2,""vid3"":2}",39.827,1,"[1051_herd, 1121_scratch, 5528_hug]",5528_hug
2,1,2024-10-08,23:16:07,2,19,middle,"{""vid1"":3,""vid2"":3,""vid3"":3}",58.960,1,"[6026_fight, 5528_hug, 4926_lead]",5528_hug
3,1,2024-10-08,23:16:07,3,39,middle,"{""vid1"":4,""vid2"":4,""vid3"":4}",82.882,1,"[5570_poke, 5790_escape, 6026_fight]",5790_escape
4,1,2024-10-08,23:16:07,4,31,right,"{""vid1"":5,""vid2"":5,""vid3"":5}",106.792,1,"[6026_fight, 1167_kiss, 5876_creep up on]",5876_creep up on
...,...,...,...,...,...,...,...,...,...,...,...
2920,86,2024-10-15,05:37:19,40,11,right,"{""vid1"":41,""vid2"":41,""vid3"":41}",1653.342,21,"[4806_flirt with, 4926_lead, 5570_poke]",5570_poke
2921,86,2024-10-15,05:37:19,41,36,middle,"{""vid1"":84,""vid2"":42,""vid3"":42}",1690.304,21,"[4789_bother, 6013_talk to, 6058_leave]",6013_talk to
2922,86,2024-10-15,05:37:19,42,2,right,"{""vid1"":43,""vid2"":43,""vid3"":86}",1718.257,21,"[4806_flirt with, 1167_kiss, 6239_pull]",6239_pull
2923,86,2024-10-15,05:37:19,43,35,right,"{""vid1"":44,""vid2"":44,""vid3"":44}",1745.103,21,"[4926_lead, 5866_follow, 1051_herd]",1051_herd


In [18]:
n = len(np.unique(cleanedTrial_stimset1.subjNum))
n

65

# StimSet2

## Subj file

In [19]:
subj_stimset2 = pd.read_csv("./raw/subj_HSvideo_stimset2.txt", delimiter="\t")
subj_stimset2.head()

,num,date,startTime,id,userAgent,endTime,duration,quizAttemptN,instrReadingTimes,quickReadingPageN,...,hiddenDurations,comments,serious,maximized,problems,gender,age,inView,viewportW,viewportH
0,4,2024-11-13,01:13:28,64204,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,01:27:46,14.302683,1,"{""0"":3.036,""1"":8.144,""2"":10.544,""3"":14.846,""4""...",0,...,NaN,NaN,1,1,NaN,F,20,True,1512,945
1,3,2024-11-13,01:13:21,67881,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,01:30:30,17.153567,1,"{""0"":1.815,""1"":3.748,""2"":6.281,""3"":9.931,""4"":1...",0,...,NaN,I hope your study goes well!! :),1,1,"The procedure was clear, however, it got very ...",F,18,True,1470,919
2,5,2024-11-13,01:16:40,68893,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,01:36:51,20.173883,1,"{""0"":1.736,""1"":4.114,""2"":8.143,""3"":12.946,""4"":...",0,...,NaN,nope,1,1,nope,F,18,True,1280,800
3,6,2024-11-13,01:26:03,68202,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,01:41:02,14.988933,1,"{""0"":1.942,""1"":3.394,""2"":5.12,""3"":8.765,""4"":12...",0,...,NaN,NaN,1,1,Nope,F,18,True,1366,768
4,7,2024-11-13,01:27:37,67941,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,01:56:05,28.473417,2,"{""0"":408.307,""1"":413.174,""2"":415.808,""3"":423.1...",0,...,NaN,NaN,1,1,NaN,M,20,True,1440,900


In [20]:
subj_stimset2.shape

(77, 21)

In [21]:
len(subj_stimset2.loc[subj_stimset2.serious == 0])

5

In [22]:
len(subj_stimset2.loc[subj_stimset2.maximized == 0])

1

In [23]:
excludedSubjNum_stimset2 = pd.concat([subj_stimset2.loc[subj_stimset2.serious == 0, "num"], subj_stimset2.loc[subj_stimset2.maximized == 0, "num"]])
len(np.unique(excludedSubjNum_stimset2))

6

## Trial file

In [24]:
trial_stimset2 = pd.read_csv("./raw/trial_HSvideo_stimset2.txt", delimiter=";")
trial_stimset2.head()

,subjNum,subjStartDate,subjStartTime,trialIndex,exptId,choicePos,vidPlayCounts,rt
0,1,2024-11-13,00:56:12,0,21,right,"{""vid1"":1,""vid2"":1,""vid3"":1}",14.434
1,1,2024-11-13,00:56:12,1,31,middle,"{""vid1"":2,""vid2"":2,""vid3"":2}",26.037
2,1,2024-11-13,00:56:12,2,6,right,"{""vid1"":3,""vid2"":3,""vid3"":3}",42.620
3,1,2024-11-13,00:56:12,3,43,left,"{""vid1"":4,""vid2"":4,""vid3"":4}",59.321
4,1,2024-11-13,00:56:12,4,0,left,"{""vid1"":10,""vid2"":5,""vid3"":5}",95.714


In [25]:
trial_stimset2.shape

(3477, 8)

In [26]:
incompleteSubjNum_stimset2 = trial_stimset2.subjNum.value_counts().loc[lambda x : x < 45].index
completeTrial_stimset2 = trial_stimset2[~trial_stimset2.subjNum.isin(incompleteSubjNum_stimset2)]

In [27]:
#drop not serious or not maximized
excSubjTrial_stimset2 = completeTrial_stimset2[~completeTrial_stimset2.subjNum.isin(excludedSubjNum_stimset2)]
len(np.unique(excSubjTrial_stimset2.subjNum)) #N

70

In [52]:
completeSubj_stimset2 = subj_stimset2[~subj_stimset2.num.isin(incompleteSubjNum_stimset2)]
cleanedSubj_stimset2 = completeSubj_stimset2[~completeSubj_stimset2.num.isin(excludedSubjNum_stimset2)]

In [29]:
excSubjTrial_stimset2 = excSubjTrial_stimset2.copy()
excSubjTrial_stimset2["exptVer"] = excSubjTrial_stimset2.apply(lambda row: find_exptVer(row, studyVer="stimset2"), axis = 1)
excSubjTrial_stimset2["options"] = excSubjTrial_stimset2.apply(lambda row: find_exptOptions(row, studyVer="stimset2"), axis = 1) 
excSubjTrial_stimset2["choice"] = excSubjTrial_stimset2.apply(lambda row: find_choice(row), axis = 1)
cleanedTrial_stimset2 = excSubjTrial_stimset2.reset_index(drop = True)
cleanedTrial_stimset2

,subjNum,subjStartDate,subjStartTime,trialIndex,exptId,choicePos,vidPlayCounts,rt,exptVer,options,choice
0,4,2024-11-13,01:13:28,0,24,right,"{""vid1"":1,""vid2"":1,""vid3"":1}",13.136,4,"[5987_creep up on, 5914_tickle, 5902_hit]",5902_hit
1,3,2024-11-13,01:13:21,0,20,left,"{""vid1"":1,""vid2"":1,""vid3"":1}",18.604,3,"[5991_examine, 6079_bother, 5875_escape]",5991_examine
2,3,2024-11-13,01:13:21,1,22,right,"{""vid1"":2,""vid2"":2,""vid3"":2}",30.488,3,"[1145_leave, 5816_ignore, 5914_tickle]",5914_tickle
3,4,2024-11-13,01:13:28,1,28,left,"{""vid1"":4,""vid2"":2,""vid3"":2}",34.081,4,"[6035_capture, 5787_accompany, 5809_pull]",6035_capture
4,3,2024-11-13,01:13:21,2,2,right,"{""vid1"":3,""vid2"":3,""vid3"":3}",50.271,3,"[5787_accompany, 5991_examine, 5875_escape]",5875_escape
...,...,...,...,...,...,...,...,...,...,...,...
3145,88,2024-11-16,22:09:32,40,31,left,"{""vid1"":41,""vid2"":41,""vid3"":41}",1255.925,23,"[5987_creep up on, 5843_huddle with, 6012_kiss]",5987_creep up on
3146,88,2024-11-16,22:09:32,41,24,right,"{""vid1"":42,""vid2"":42,""vid3"":42}",1271.683,23,"[5878_follow, 6004_herd, 5816_ignore]",5816_ignore
3147,88,2024-11-16,22:09:32,42,32,middle,"{""vid1"":43,""vid2"":43,""vid3"":43}",1347.516,23,"[5814_talk to, 6079_bother, 4397_hug]",6079_bother
3148,88,2024-11-16,22:09:32,43,35,left,"{""vid1"":44,""vid2"":44,""vid3"":44}",1361.882,23,"[6017_encircle, 1145_leave, 1012_push]",6017_encircle


In [30]:
n_stimset2 = len(np.unique(cleanedTrial_stimset2.subjNum))
n_stimset2

70

# Offdiagonal

## Subj file

In [31]:
subj_offdiagonal = pd.read_csv("./raw/subj_HSvideo_offdiagonal.txt", delimiter="\t")
subj_offdiagonal.head()

,num,date,startTime,id,userAgent,endTime,duration,quizAttemptN,instrReadingTimes,quickReadingPageN,...,hiddenDurations,comments,serious,maximized,problems,gender,age,inView,viewportW,viewportH
0,1,2025-11-19,09:13:50,72861,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,09:38:41,24.853450,1,"{""0"":1.511,""1"":10.586,""2"":13.942,""3"":16.764,""4...",0,...,NaN,Nope,1,1,Nope it was clear.,M,21,True,1920,1080
1,2,2025-11-19,09:46:48,68691,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,10:03:34,16.758450,1,"{""0"":2.013,""1"":5.694,""2"":9.592,""3"":12.927,""4"":...",0,...,NaN,No,1,1,No,M,20,True,1470,920
2,3,2025-11-19,16:08:27,72830,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,16:27:32,19.084200,1,"{""0"":3.641,""1"":7.557,""2"":12.791,""3"":20.874,""4""...",0,...,NaN,This was fun!,1,1,"On some parts, one of the videos wouldn't be v...",N,19,True,1470,923
3,4,2025-11-19,16:11:10,68258,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,16:32:54,21.746517,1,"{""0"":3.314,""1"":6.73,""2"":9.013,""3"":15.915,""4"":1...",0,...,"0.625,1.642,247.829",<br />no,1,1,No,F,20,True,1440,900
4,5,2025-11-19,16:14:58,72163,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,16:34:27,19.481750,1,"{""0"":1.776,""1"":2.599,""2"":6.87,""3"":16.107,""4"":2...",0,...,NaN,"This was honestly a pretty fun experiment, tha...",1,1,A few times I accidentally selected the option...,M,21,True,1707,1067


In [32]:
subj_offdiagonal.shape

(501, 21)

In [33]:
len(subj_offdiagonal.loc[subj_offdiagonal.serious == 0])

34

In [34]:
len(subj_offdiagonal.loc[subj_offdiagonal.maximized == 0])

18

In [35]:
excludedSubjNum_offdiagonal = pd.concat([subj_offdiagonal.loc[subj_offdiagonal.serious == 0, "num"], subj_offdiagonal.loc[subj_offdiagonal.maximized == 0, "num"]])
len(np.unique(excludedSubjNum_offdiagonal))

48

## Trial file

In [36]:
trial_offdiagonal = pd.read_csv("./raw/trial_HSvideo_offdiagonal.txt", delimiter=";")
trial_offdiagonal.head()

,subjNum,subjStartDate,subjStartTime,trialIndex,exptId,choicePos,vidPlayCounts,rt
0,1,2025-11-19,09:13:50,0,15,right,"{""vid1"":1,""vid2"":1,""vid3"":1}",19.521
1,1,2025-11-19,09:13:50,1,43,left,"{""vid1"":4,""vid2"":2,""vid3"":2}",41.227
2,1,2025-11-19,09:13:50,2,20,right,"{""vid1"":3,""vid2"":3,""vid3"":6}",61.208
3,1,2025-11-19,09:13:50,3,18,right,"{""vid1"":4,""vid2"":4,""vid3"":4}",77.304
4,1,2025-11-19,09:13:50,4,4,middle,"{""vid1"":5,""vid2"":5,""vid3"":5}",94.014


In [37]:
len(np.unique(trial_offdiagonal.subjNum))

534

In [38]:
incompleteSubjNum_offdiagonal = trial_offdiagonal.subjNum.value_counts().loc[lambda x : x < 45].index
completeSubj_offdiagonal = subj_offdiagonal[~subj_offdiagonal.num.isin(incompleteSubjNum_offdiagonal)]

In [39]:
completeTrial_offdiagonal = trial_offdiagonal[~trial_offdiagonal.subjNum.isin(incompleteSubjNum_offdiagonal)]
excSubjTrial_offdiagonal = completeTrial_offdiagonal[~completeTrial_offdiagonal.subjNum.isin(excludedSubjNum_offdiagonal)]

In [53]:
cleanedSubj_offdiagonal = completeSubj_offdiagonal[~completeSubj_offdiagonal.num.isin(excludedSubjNum_offdiagonal)]

In [41]:
excSubjTrial_offdiagonal = excSubjTrial_offdiagonal.copy()
excSubjTrial_offdiagonal["exptVer"] = excSubjTrial_offdiagonal.apply(lambda row: find_exptVer(row, studyVer="offdiagonal"), axis = 1)
excSubjTrial_offdiagonal["options"] = excSubjTrial_offdiagonal.apply(lambda row: find_exptOptions(row, studyVer="offdiagonal"), axis = 1) 
excSubjTrial_offdiagonal["choice"] = excSubjTrial_offdiagonal.apply(lambda row: find_choice(row), axis = 1)

In [42]:
cleanedTrial_offdiagonal = excSubjTrial_offdiagonal.reset_index(drop = True)
trialNoSubj_offdiagonal = list(set(completeTrial_offdiagonal.subjNum) - set(completeSubj_offdiagonal.num))
cleanedTrial_offdiagonal = cleanedTrial_offdiagonal[~cleanedTrial_offdiagonal.subjNum.isin(trialNoSubj_offdiagonal)]
cleanedTrial_offdiagonal

,subjNum,subjStartDate,subjStartTime,trialIndex,exptId,choicePos,vidPlayCounts,rt,exptVer,options,choice
0,1,2025-11-19,09:13:50,0,15,right,"{""vid1"":1,""vid2"":1,""vid3"":1}",19.521,1,"[5948_flirt with, 6026_fight, 4397_hug]",4397_hug
1,1,2025-11-19,09:13:50,1,43,left,"{""vid1"":4,""vid2"":2,""vid3"":2}",41.227,1,"[5986_fight, 6017_encircle, 6013_talk to]",5986_fight
2,1,2025-11-19,09:13:50,2,20,right,"{""vid1"":3,""vid2"":3,""vid3"":6}",61.208,1,"[6013_talk to, 5948_flirt with, 4861_throw]",4861_throw
3,1,2025-11-19,09:13:50,3,18,right,"{""vid1"":4,""vid2"":4,""vid3"":4}",77.304,1,"[5870_poke, 5866_follow, 5986_fight]",5986_fight
4,1,2025-11-19,09:13:50,4,4,middle,"{""vid1"":5,""vid2"":5,""vid3"":5}",94.014,1,"[5866_follow, 5878_follow, 5875_escape]",5878_follow
...,...,...,...,...,...,...,...,...,...,...,...
20335,592,2026-01-14,04:52:09,40,17,middle,"{""vid1"":82,""vid2"":41,""vid3"":41}",1355.120,164,"[6012_kiss, 1121_scratch, 5849_approach]",1121_scratch
20336,592,2026-01-14,04:52:09,41,5,right,"{""vid1"":42,""vid2"":42,""vid3"":42}",1377.055,164,"[6034_avoid, 5471_ignore, 6005_throw]",6005_throw
20337,592,2026-01-14,04:52:09,42,34,left,"{""vid1"":86,""vid2"":43,""vid3"":43}",1417.049,164,"[4793_avoid, 5814_talk to, 5528_hug]",4793_avoid
20338,592,2026-01-14,04:52:09,43,27,middle,"{""vid1"":88,""vid2"":132,""vid3"":132}",1468.419,164,"[5870_poke, 6239_pull, 5843_huddle with]",6239_pull


In [43]:
n_offdiagonal = len(np.unique(cleanedTrial_offdiagonal.subjNum))
n_offdiagonal

450

# Combine 

## Subj file

In [54]:
# distinguish subjs across studies
cleanedSubj_stimset1["num"] = cleanedSubj_stimset1["num"]
cleanedSubj_stimset2["num"] = cleanedSubj_stimset2["num"] + 1000
cleanedSubj_offdiagonal["num"] = cleanedSubj_offdiagonal["num"] + 2000

/var/folders/k7/mm5crvhx3210xfdblpr8gzsw0000gn/T/ipykernel_34098/948422539.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleanedSubj_stimset1["num"] = cleanedSubj_stimset1["num"].copy()
/var/folders/k7/mm5crvhx3210xfdblpr8gzsw0000gn/T/ipykernel_34098/948422539.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleanedSubj_stimset2["num"] = cleanedSubj_stimset2["num"].copy() + 1000
/var/folders/k7/mm5crvhx3210xfdblpr8gzsw0000gn/T/ipykernel_34098/948422539.py:4: SettingWithCopyWarning: 
A value is tryi

In [55]:
subj_all = pd.concat([cleanedSubj_stimset1, cleanedSubj_stimset2, cleanedSubj_offdiagonal], axis=0, ignore_index=True)
subj_all

,num,date,startTime,id,userAgent,endTime,duration,quizAttemptN,instrReadingTimes,quickReadingPageN,...,hiddenDurations,comments,serious,maximized,problems,gender,age,inView,viewportW,viewportH
0,1,2024-10-08,23:16:07,67846,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,23:35:53,19.752700,1,"{""0"":3.921,""1"":6.676,""2"":10.601,""3"":15.68,""4"":...",0,...,NaN,NaN,1,1,N/A. I just had difficulty finding the odd one...,M,20,True,1536,864
1,3,2024-10-08,23:19:39,67851,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,23:59:15,39.596517,1,"{""0"":0.956,""1"":1.671,""2"":2.621,""3"":7.022,""4"":1...",0,...,NaN,NaN,1,1,None of it was unclear but at times it was har...,M,18,True,1440,900
2,4,2024-10-08,23:25:15,67998,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,00:08:53,43.629583,1,"{""0"":1.138,""1"":3.412,""2"":5.454,""3"":8.593,""4"":1...",0,...,2.545,NaN,1,1,NaN,F,18,True,1440,900
3,6,2024-10-08,23:57:45,63805,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,00:24:21,26.614033,1,"{""0"":4.676,""1"":7.235,""2"":71.03,""3"":79.243,""4"":...",0,...,NaN,NaN,1,1,NaN,F,20,True,1512,945
4,7,2024-10-09,00:08:22,67795,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,00:34:07,25.745483,1,"{""0"":32.602,""1"":36.445,""2"":38.315,""3"":43.502,""...",0,...,"0.699,0.019,13.738",NaN,1,1,No,F,21,True,1470,919
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
580,2586,2026-01-14,02:02:19,73578,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,02:23:12,20.878767,1,"{""0"":2.417,""1"":8.342,""2"":11.748,""3"":16.348,""4""...",0,...,NaN,No!,1,1,"No, everything was quite clear, and there were...",M,21,True,1707,1067
581,2587,2026-01-14,02:32:46,69908,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,02:51:23,18.618117,1,"{""0"":3.431,""1"":7.445,""2"":10.485,""3"":23.449,""4""...",0,...,NaN,"nope, pretty interesting i was attributing per...",1,1,Nope,F,21,True,1194,777
582,2588,2026-01-14,02:51:55,68110,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,03:11:51,19.924750,1,"{""0"":2.123,""1"":5.156,""2"":7.82,""3"":11.51,""4"":16...",0,...,NaN,No.,1,1,"Nothing, everything was clear.",F,19,True,1280,800
583,2590,2026-01-14,03:07:48,73134,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,03:42:25,34.605367,1,"{""0"":1.571,""1"":2.518,""2"":3.286,""3"":17.268,""4"":...",0,...,"1.215,63.801,28.656,5.988,25.587,19.572,36.151...",Nope,1,1,No issues,M,21,True,1470,831


In [58]:
n = len(np.unique(subj_all.num))
subj_all.to_csv(f"{n}subj_cleaned_subj_HSvideo.csv", sep = "\t", index = False, header = True)

## Trial file

In [47]:
# distinguish subjs across studies
cleanedTrial_stimset1["subjNum"] = cleanedTrial_stimset1["subjNum"] 
cleanedTrial_stimset2["subjNum"] = cleanedTrial_stimset2["subjNum"] + 1000
cleanedTrial_offdiagonal["subjNum"] = cleanedTrial_offdiagonal["subjNum"] + 2000

In [59]:
trial_all = pd.concat([cleanedTrial_stimset1, cleanedTrial_stimset2, cleanedTrial_offdiagonal], axis=0, ignore_index=True)
trial_all

,subjNum,subjStartDate,subjStartTime,trialIndex,exptId,choicePos,vidPlayCounts,rt,exptVer,options,choice
0,1,2024-10-08,23:16:07,0,29,right,"{""vid1"":1,""vid2"":1,""vid3"":1}",21.897,1,"[5570_poke, 4806_flirt with, 5901_huddle with]",5901_huddle with
1,1,2024-10-08,23:16:07,1,34,right,"{""vid1"":2,""vid2"":2,""vid3"":2}",39.827,1,"[1051_herd, 1121_scratch, 5528_hug]",5528_hug
2,1,2024-10-08,23:16:07,2,19,middle,"{""vid1"":3,""vid2"":3,""vid3"":3}",58.960,1,"[6026_fight, 5528_hug, 4926_lead]",5528_hug
3,1,2024-10-08,23:16:07,3,39,middle,"{""vid1"":4,""vid2"":4,""vid3"":4}",82.882,1,"[5570_poke, 5790_escape, 6026_fight]",5790_escape
4,1,2024-10-08,23:16:07,4,31,right,"{""vid1"":5,""vid2"":5,""vid3"":5}",106.792,1,"[6026_fight, 1167_kiss, 5876_creep up on]",5876_creep up on
...,...,...,...,...,...,...,...,...,...,...,...
26320,2592,2026-01-14,04:52:09,40,17,middle,"{""vid1"":82,""vid2"":41,""vid3"":41}",1355.120,164,"[6012_kiss, 1121_scratch, 5849_approach]",1121_scratch
26321,2592,2026-01-14,04:52:09,41,5,right,"{""vid1"":42,""vid2"":42,""vid3"":42}",1377.055,164,"[6034_avoid, 5471_ignore, 6005_throw]",6005_throw
26322,2592,2026-01-14,04:52:09,42,34,left,"{""vid1"":86,""vid2"":43,""vid3"":43}",1417.049,164,"[4793_avoid, 5814_talk to, 5528_hug]",4793_avoid
26323,2592,2026-01-14,04:52:09,43,27,middle,"{""vid1"":88,""vid2"":132,""vid3"":132}",1468.419,164,"[5870_poke, 6239_pull, 5843_huddle with]",6239_pull


In [60]:
n = len(np.unique(trial_all.subjNum))
trial_all.to_csv(f"{n}subj_cleaned_trial_HSvideo.csv", sep = ";", index = False, header = True)